# Tutorial for DTM module

## About the Document-Term Matrix
A document-term matrix (DTM) is the standard interface for analysis and information of document data. It consists in its raw form of a list of token counts per document in the corpus. Each unique token form is called a term. Thus it is really a list of term counts per document, arranged as matrix.
In the Lexos App, sklearn's `CountVectorizer` is used to produce the DTM. In the Lexos API, Textacy's `Vectorizer` is the default vectorizer.

A DTM is fundamental in text analysis because it transforms a collection of documents into a structured, numerical format suitable for computational analysis, as it enables a wide range of analyses, such as identifying common terms, comparing document similarity, and serving as input for machine learning models. By quantifying textual data, the DTM makes it possible to apply statistical and algorithmic techniques to extract insights from text.

## 1. Prerequisites

- Python ≥ 3.8
- `lexos` installed (see main README)
- NLP model for spaCy (if processing raw text, see main README)

## 2. Preparing Your Documents

You can supply documents in two formats to `DTM`:

### 2.1 Using spaCy `Doc` objects

spaCy `Doc` objects represent processed documents, containing tokens, linguistic annotations, and other useful metadata. The `DTM` module can directly accept lists of spaCy `Doc` objects as input, allowing you to leverage spaCy's robust tokenization and preprocessing capabilities. This is especially useful when you want to include linguistic features such as lemmatization, part-of-speech tagging, or custom token filtering before building your document-term matrix. Simply process your raw texts with spaCy and pass the resulting `Doc` objects to the `DTM` for analysis.

In [21]:
import spacy
from lexos.dtm import DTM

nlp = spacy.load("en_core_web_sm")
docs = [nlp("This is the first document."), nlp("Here is the second.")]
labels = ["Doc1", "Doc2"]

### 2.2 Using lists of token strings

Alternatively, you can provide your documents as lists of token strings, where each document is represented as a list of its tokens (words). This approach is useful if you have already preprocessed your text (e.g., tokenized, cleaned, or filtered) and want to bypass spaCy's processing pipeline. Simply pass a list of these token lists to the `DTM` module, along with optional document labels. This method offers flexibility and allows you to integrate custom preprocessing workflows with the DTM functionality.

In [22]:
docs = [["token1", "token2", "token10"], ["another", "set", "of", "tokens", "token10"]]
labels = ["T1", "T2"]

## 3. Building the DTM

The following code demonstrates how to build a Document-Term Matrix (DTM) using the `DTM` class from the Lexos library. By providing your preprocessed documents and optional labels, you can generate a matrix where each row represents a document and each column corresponds to a unique term across all documents. The resulting DTM enables further analysis, such as inspecting term frequencies and converting the matrix to a DataFrame for visualization or export.

In [23]:
from lexos.dtm import DTM, Vectorizer

dtm = DTM(vectorizer=Vectorizer())
dtm(docs=docs, labels=labels)

# Check shape: (n_docs, n_terms)
# Note that the count of terms is the count of UNIQUE terms across all documents
# and not the total number of terms.
print(dtm.doc_term_matrix.shape)

# Check labels
print(dtm.labels)
# Check docs
print(dtm.docs)

(2, 7)
['T1', 'T2']
[['token1', 'token2', 'token10'], ['another', 'set', 'of', 'tokens', 'token10']]


## 4. Inspecting Terms and Counts

To better understand the structure of your DTM, you can inspect the sorted list of unique terms (vocabulary) and their corresponding counts across all documents. The cell below demonstrates how to access the sorted list of terms using `dtm.sorted_terms_list` and retrieve a mapping of each term to its total count with `dtm.sorted_term_counts`. This allows you to quickly review which terms are present in your corpus and how frequently they appear, providing valuable insights before further analysis or visualization.

In [24]:
# Sorted list of terms
terms = dtm.sorted_terms_list
print(terms)

# Term counts mapping
counts = dtm.sorted_term_counts
print(counts)

['another', 'of', 'set', 'token1', 'token2', 'token10', 'tokens']
{'another': 0, 'of': 1, 'set': 2, 'token1': 3, 'token2': 5, 'token10': 4, 'tokens': 6}


## 5. Converting to a DataFrame

The cell below demonstrates how to convert the DTM object into a pandas DataFrame using the `to_df()` method. This allows for easier inspection, manipulation, and export of the document-term matrix, making it suitable for further analysis or visualization in pandas.

In [25]:
import pandas as pd

# Basic DataFrame conversion
df = dtm.to_df()
df

,T1,T2
another,0,1
of,0,1
set,0,1
tokens,0,1
token1,1,0
token10,1,1
token2,1,0


### Customizing the DataFrame Output

You can tailor the output of your document-term matrix DataFrame using several options in the `to_df()` method:

- **Sort by a specific column:**  
    Use the `by` parameter to sort the DataFrame by a particular document label (e.g., `by="T2"`). By default, the DataFrame is sorted by the first label.

- **Control sort order:**  
    Set `ascending=False` to sort in descending order (largest to smallest values). The default is `ascending=True` for ascending order.

- **Display percentages instead of raw counts:**  
    Set `as_percent=True` to show term frequencies as percentages of the total terms in each document, rather than raw counts. This is useful for comparing documents of different lengths.

- **Adjust decimal precision:**  
    Use the `rounding` parameter to specify the number of decimal places for percentages (e.g., `rounding=2` for two decimals). The default is 1 decimal place.

- **Add row statistics:**  
    Include summary statistics for each term across all documents by setting `sum=True` (total count), `mean=True` (average count), or `median=True` (median count). These columns help you quickly identify the most common or distinctive terms.

- **Transpose the matrix:**  
    Set `transpose=True` to swap rows and columns, so that documents become columns and terms become rows. This can be helpful for certain types of analysis or visualization.

These options allow you to customize the DataFrame to best suit your analysis and reporting needs.

#### Example with percentages with totals and two decimals

This example demonstrates how to convert the document-term matrix to a DataFrame that displays term frequencies as percentages, rounded to two decimal places, and includes a total count column for each term. This format makes it easy to compare term usage across documents of different lengths and quickly identify the most frequent terms in the corpus.

In [26]:
df_pct = dtm.to_df(
    as_percent=True,
    rounding=2,
    sum=True
)
df_pct

,T1,T2,Total
another,0.0,12.5,12.5
of,0.0,12.5,12.5
set,0.0,12.5,12.5
tokens,0.0,12.5,12.5
token1,12.5,0.0,12.5
token10,12.5,12.5,25.0
token2,12.5,0.0,12.5


## 6. Running the Test Suite

Also included in this repository is a comprehensive test suite of the dtm module. It can be run with the following command:
```bash
uv run pytest tests/dtm
```

The test suite was found to have 100% coverage. This can be verified with the following commands:
```bash
uv run pytest --cov=src/lexos/dtm --cov-report=html tests/dtm
```
